# Introduction

Recommender systems are widely used by mainstream media platforms to generate new options that fill your watchlist, often employing complex machine learning algorithms. However, there’s a simpler approach that utilizes web scraping to approach similar results. In this tutorial, I will guide you through the process of building a movie and TV show recommender system using a web scraper. By exploring The Movie Database, I will uncover how to extract valuable information about shared cast members and leverage this data to generate tailored recommendations.

## 1. Set Up the Project

Open the Terminal and execute the following commands to create a new Scrapy project named `TMDB_scraper` in your current directory:
```bash
conda activate PIC16B-24F
scrapy startproject TMDB_scraper
cd TMDB_scraper
```
This sequence activates your `PIC16B-24F` environment, initializes the Scrapy project, and navigates into it.

We need to guide our spider with written instructions. To set up these web-scraping instructions, we’ll create a file named `tmdb_spider.py` inside the directory `TMDB_scraper/TMDB_scraper/spiders`. In this file, we’ll begin the implementation by importing the Scrapy package and defining the `TmdbSpider` class, which inherits from `scrapy.Spider`. Since we will later use `urljoin` from `urllib.parse` to handle URLs, we’ll also import it here.
```python
import scrapy
from urllib.parse import urljoin

class TmdbSpider(scrapy.Spider):
    name = 'tmdb_spider'
    def __init__(self, subdir="", *args, **kwargs):
        self.start_urls = [f"https://www.themoviedb.org/movie/{subdir}/"]
```

## 2. Scraper Implementation

To break down the scraper step-by-step, we’ll start from the main page of your selected favorite movie. For example, if your favorite movie is Interstellar, its URL on The Movie Database (TMDB) is: https://www.themoviedb.org/movie/157336-interstellar. From this main page, we will navigate to the “Full Cast & Crew” section. On the "Full Cast & Crew" page, we will follow each cast member’s individual link to retrieve the movies and TV shows they have previously acted in. After collecting this information for all the cast members, we’ll analyze the scraped data to identify the movies and TV shows with the highest number of shared actors, which will serve as the basis for our recommendations.

### 2.1 parse Function

In the first section, we’ll build a function named `parse` to handle navigation from the main page of the selected movie to its “Full Cast & Crew” page. By examining the URL pattern for the “Full Cast & Crew” page—for instance, https://www.themoviedb.org/movie/157336-interstellar/cast—we see that it consists of the main page URL with `"/cast"` appended to it. <br>
In this function, the main page URL is provided as an argument to the function. We then modify this URL by appending `"/cast"` and pass the updated URL to the next function for further processing and yielding results. The code implementation is shown below:
```python
def parse(self, response):
        """
        Yields a Scrapy request for the Full Cast & Crew page 
        of the film handle by parse_full_credits()
        Args: 
            response: Response object, the main page of the movie
        Yield:
            A scrapy request to Full Cast & Crew page
        """
        # Obtain Full Cast & Crew page url
        cast_url = response.url + "/cast"
        # Yield a request to Full Cast & Crew page
        yield scrapy.Request(cast_url, callback = self.parse_full_credits)
```

### 2.2 parse_full_credits Function

In the next function, we will navigate from the "Full Cast & Crew" page to each cast member’s individual profile page. We’ll start by providing the "Full Cast & Crew" page link as an argument to the function. To experiment with this process, open Developer Tools by pressing `Command + Option + I`. With Developer Tools open, inspect the page layout and locate the elements that contain `href` links to each cast member’s profile page. <br> <br>
   On the "Full Cast & Crew" page, each cast member’s details, including a link to their profile page, are wrapped in `<li>` elements, which represent individual list items. The `<p class="character">` element within each `<li> `provides additional detail to verify the selected `<li>` is a cast member. By selecting each `<a href>` within these specific `<li>` elements, we can gather the URLs to each cast member's profile page, allowing our scraper to navigate to these pages and retrieve further details.

Using the CSS selector `li:has(p.character) > a::attr(href)` achieve the above purpose:
- `li:has(p.character)` targets `<li>` elements that contain `<p class="character">`, ensuring only relevant list items are selected.
- `a::attr(href)` then selects the href attribute from the `<a>` tag inside the `<li>`, extracting each cast member’s profile page URL. <br>

Before writing the CSS selector in the function, we can experiment it on a `scrapy shell`. <br>
Open terminal and type:
```bash
conda activate PIC16B-24F
cd TMDB_scraper
```
Try to change user agent before run into `403 Forbidden` erros:
```bash
scrapy shell -s USER_AGENT = 
  'Scrapy/2.8.0 (+https://scrapy.org)' https://www.themoviedb.org/...
```
Then we open a `scrapy shell` for the "Full Cast & Crew" Page:
```bash
scrapy shell https://www.themoviedb.org/movie/157336-interstellar/cast
```
Then we experiment with the CSS selector:
```bash
cast = response.css('li:has(p.character) > a::attr(href)').getall()
cast
```

After running this CSS selector, you’ll see that we’ve obtained the paths for each cast member’s individual page. For our function, we need the complete URL to pass along and operate on. To achieve this, we’ll need to join each path with the domain.

To summarize the function’s flow: we’ll start by initializing the website’s domain. Then, we’ll use a CSS selector to retrieve the paths to each cast member’s page. Next, we’ll join each path with the domain to form full URLs and pass them to the `parse_actor_page()` function to retrieve the relevant movies and TV shows for each actor.

Here is the complete version of the `parse_full_credits()` function:
```python
def parse_full_credits(self, response):
        """
        Yields a Scrapy request for each cast's page 
        from the cast page handle by parse_actor_page()
        Args: 
            response: Response object, the Full Cast & Crew page
        Yield:
            Scrapy requests for each cast's individual page
        """
        # Define a base_url
        base_url = "https://www.themoviedb.org/"
        
        # Extract individual cast's URL path
        casts = (set
                (response.css
                ('li:has(p.character) > a::attr(href)').getall()))
        
        # Trace each cast's URL and yield a request for each url 
        for cast in casts:
            yield 
            scrapy.Request(urljoin(base_url, cast), 
                           callback = self.parse_actor_page)
```

### 2.3 parse_actor_page Function

The final function implementation focuses on retrieving all the movies and TV shows in which the actor has appeared. At this stage, the actor’s page is passed as an argument to the function. The function then yields a dictionary where the keys are the actor’s name, and the values are each individual work they have performed.
To prepare, we’ll open the developer tools to inspect the HTML structure of the actor’s profile page. Note that the credit list contains multiple sections, such as “Acting” and “Production.” We’ll limit our focus to the “Acting” section and filter out other sections to stay relevant. Let’s begin writing the CSS selector step by step. First, open a new Scrapy shell to experiment:
```bash
scrapy shell https://www.themoviedb.org/person/83002-jessica-chastain
```

To start, we need to capture the actor’s name. According to the HTML structure, the actor’s name is embedded in the page’s title. We can retrieve the title using the following CSS selector:
```bash
title = response.css('title::text').get()
title
```

Upon printing the title text, you’ll notice that it includes not only the actor’s name but also the suffix “ - The Movie Database (TMDB).” We’ll need to account for this by using substring techniques to isolate just the actor’s name. Next, we’ll proceed to extract the list of movies and TV shows the actor has appeared in within the “Acting” section.

By selecting the movie or show titles from this specific "Acting" section `h3:contains("Acting")` within the credit list, we focus specifically on acting roles. Within the "Acting" section, each movie or TV show the actor has appeared in is listed within a `<table>` element, which includes the title and role. Using the CSS selector `response.css('.credits_list h3:contains("Acting") + table .role a.tooltip bdi::text').getall()` achieves our purpose: <br>
-`.credits_list h3:contains("Acting")` selects the `<h3>` element labeled "Acting" within the credits list <br>
-`table` then selects the `<table>` element following "Acting" header, which contains the relevant acting credits. <br>
-`.role a.tooltip bdi::text` targets the specific text within each `<bdi>` element inside an `<a>` tag with the `.tooltip` class, isolating the titles of each movie or TV show where the actor appeared. <br>
Test the above CSS selector with:
```bash
response.css('.credits_list h3:contains("Acting") 
            + table .role a.tooltip bdi::text').getall())
```

After successfully testing all the CSS selectors and gathering the necessary information, we proceed to implement the function. We begin by selecting the title of the actor’s page, then use a substring function to extract the actor's name. Next, we apply the CSS selector mentioned earlier to retrieve the movies and TV shows in which the actor appeared. Finally, we yield the results as a dictionary, recording the actor's name alongside each work. The function appears as follows:
```python
def parse_actor_page(self, response):
        """
        Yields a dictionary for each movie/TV show in which the actor has acted
        The key is the actor's name and the value is the unique name of the work
        Args: 
            response: Response object, the actor's page
        Yield:
            A dictionary with keys as actor's name and movie/TV title
        """
        # Extract the text from the <title> tag
        title = response.css('title::text').get()

        # Extract actor's name from the title
        actor_name = title.split("—")[0].strip()
        
        # Extract the unique title of movie/TV shows
        works = set(
              response.css
              ('.credits_list h3:contains("Acting")+table .role a.tooltip bdi::text')
              .getall())
    
        # Yield a dictionary with key = actor name and value = title of movie/TV shows
        for work in works:
            yield {"actor": actor_name, "movie_or_TV_name": work.strip()}
```

## 3. Make Recommendations

Now, let’s output the results into a CSV file. Open your terminal and run the following code:
```bash
conda activate PIC16B-24F
cd TMDB_scraper
scrapy crawl tmdb_spider -o results.csv -a subdir=157336-interstellar
```
You will discover a generated `results.csv` file inside the project folder. Next, move the resulting CSV file to the same directory as your project files for further operations. We’ll then import the CSV file within our script to continue processing the data. We’ll then rename the columns for clarity, and remove any rows containing the original film that was input.

In [1]:
import pandas as pd

In [2]:
# Import the Dataframe
df = pd.read_csv('results.csv')

In [3]:
# Check the Dataset Imported Correctly
df.head()

,actor,movie_or_TV_name
0,Kristian Van der Heyden,Interstellar
1,Kristian Van der Heyden,Zone Stad
2,Kristian Van der Heyden,Spoed
3,William Patrick Brown,Interstellar
4,William Patrick Brown,Mighty Med


In [4]:
# Rename the Columns of the Dataset
df.columns = ['Actor', 'Movie_or_TV_Show']

In [5]:
# Remove the Original Input Film from the Table
df = df[df['Movie_or_TV_Show'] != 'Interstellar']

We create a recommender table to count the shared cast members for each movie and TV show. Using the `groupby` method along with the `size()` function, we count the occurrences of each unique movie or TV show. A new column named "Count" is created to store these counts, and finally, we sort the DataFrame based on the values in the "Count" column. The top values represent the most recommended titles based on shared cast members.

In [12]:
# Generate a Recommender Table for Shared Cast Movies
recommend = (df.groupby('Movie_or_TV_Show')
             .size()
             .reset_index(name = 'Count')
             .sort_values(by='Count', ascending=False))
recommend.head()

,Movie_or_TV_Show,Count
1115,Saturday Night Live,30
1643,The View,27
721,Late Night with Seth Meyers,27
631,Inside 'Interstellar',24
1631,The Tonight Show Starring Jimmy Fallon,24


In [7]:
print("The Top 5 Recommended Films based on Your Interest in Interstellar are:")
for i in range(5):
    print(recommend['Movie_or_TV_Show'].iloc[i])

The Top 5 Recommended Films based on Your Interest in Interstellar are:
Saturday Night Live
The View
Late Night with Seth Meyers
Inside 'Interstellar'
The Tonight Show Starring Jimmy Fallon


## 4. Visualize Recommendations

In this section, we will visualize the recommendations using a bar plot. We will display the top 5 recommendations, with a tie among 18 rated movies or TV shows. The plotting package we will use is Plotly.

In [8]:
import plotly.express as px
import plotly.io as pio
pio.renderers.default="iframe"

In [9]:
# Retrieve Top 10 Recommendations
Top_18 = recommend.head(18)

In [10]:
Top_18 = Top_18.sort_values(by = 'Count', ascending = True)
Top_18

,Movie_or_TV_Show,Count
290,Conan,18
1371,The Dark Knight Rises,18
672,Jimmy Kimmel Live!,18
719,Late Night with Conan O'Brien,18
1765,Watch What Happens Live with Andy Cohen,18
1368,The Daily Show,18
709,LIVE with Kelly and Mark,21
1749,Variety Studio: Actors on Actors,21
1433,The Graham Norton Show,21
1561,The Oscars,21


In [12]:
fig = px.bar(Top_18, 
             x = 'Count', 
             y = 'Movie_or_TV_Show', 
             color = 'Count',
             color_continuous_scale='PiYG_r',
             text='Count',
             title = 'Top Picks: Movies and Shows with Common Actors', 
             labels = {'Movie_or_TV_Show': 'Movie or TV Show', 
                       'Count': 'Shared Cast Number'})
fig.update_layout(plot_bgcolor = '#FFD6DF')
fig.show()

## Conclusion

Overall, this is how we utilize web scraping to create a recommender system based on a movie or TV show. My favorite movie, "Interstellar", has recommendations such as "Saturday Night Live," "The View," "Late Night with Seth Meyers," "Inside Interstellar," and "The Tonight Show Starring Jimmy Fallon." Considering my fondness for "Saturday Night Live," I believe this is a reliable recommendation system. I hope you successfully implement yours. Happy spider scraping!